# Getting set up — your own Earth Engine access

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/00_setup.ipynb)

This notebook is for students: it takes you from a plain Google account to a
rendered satellite animation running under **your own** Earth Engine project, in
about fifteen minutes. Every later notebook in this series assumes you have done
this once.

Why "your own"? Earth Engine bills compute quota to a *project*. If everyone in a
course rendered under one shared project, nobody could tell whose job ate the
quota — and you could not take your work with you afterwards. Your project is
free (noncommercial use), takes five minutes to create, and is yours.

## 1 · What you need

A Google account. Nothing else — no credit card, no local install (this runs in
Colab), no key files.

Two things get created, and it helps to keep them apart:

- an **Earth Engine registration** — Google approving your account for EE's
  noncommercial tier, and
- a **Google Cloud project** — the named container your usage is accounted to.
  Its *project id* is the one string you will paste into every notebook here.

## 2 · Create your own Earth Engine project

1. Open [code.earthengine.google.com/register](https://code.earthengine.google.com/register)
   and sign in with your Google account.
2. Choose **Register a Noncommercial or Commercial Cloud project**.
3. Pick **Unpaid usage**, and *Academia & Research* (you are a student — that is
   exactly who this tier is for).
4. **Create a new Google Cloud Project.** Choose any project id you like, e.g.
   `ee-yourname`. No organization is fine.
5. Confirm. The summary page ends with *"Your project is registered"*.

**The id you just chose is your `PROJECT` everywhere in this series; never use
someone else's.** If a notebook ever asks you for a project id, it means this
one.

## 3 · Install

In Colab the next cell installs the package; locally it does nothing (install
once into an environment instead — see below).

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code
# pip resolves by version (0.2.0), which doesn't change per commit on a branch --
# --force-reinstall pulls the latest commit. After a code update: Runtime ->
# Restart session, then Run all, so the new module is re-imported.

Running **locally** instead of Colab? Once, in a terminal:

```bash
conda create -n gee python=3.11 && conda activate gee
pip install "gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@main"
```

## 4 · Authenticate

Fill in your project id (from §2), then authenticate — a one-time step per
machine. In Colab, `auth.init` runs `ee.Authenticate()`, which opens a Google
sign-in window: approve it and you are done. Locally, run `earthengine authenticate` once
in a terminal first; after that `auth.init` just works.

If the sign-in flow fails halfway, just run the cell again.

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"
AOI = [13.75, 52.97, 13.82, 53.01]           # [west, south, east, north] — Grumsin beech forest, ~5 km
START, END = "2025-05-01", "2025-08-01"      # short on purpose: a fork should render in minutes

In [ ]:
from gee_animation import auth

auth.init(PROJECT)

## 5 · The smoke test

One call: a cloud-masked, monthly-median NDVI animation over the Grumsin beech
forest (a UNESCO World Heritage beech reserve north-east of Berlin), three summer
months, rendered at 720p. It fetches live Earth Engine imagery, so it takes a
couple of minutes the first time; everything is cached after that.

In [ ]:
from gee_animation.api import animate

anim = animate(AOI, sensor="sentinel2", index="ndvi", start=START, end=END,
               project=PROJECT, preset="720p", name="setup_smoke", out_dir="out")
anim      # plays inline: ~3 monthly frames over the Grumsin beech forest

If a video plays above this line, you are set up: the imagery was fetched,
composited, colorized and encoded under your project. Every notebook after this
one builds on exactly that call.

## 6 · When it does not work

Two failures account for nearly every problem:

- **`not signed up for Earth Engine or project is not registered`** — this is
  *not* an authentication problem. Either the registration from §2 has not
  finished yet (it is usually instant, occasionally hours), or `PROJECT` names a
  Cloud project that was never EE-registered. Re-check §2 step 5, and that the id
  is spelled exactly.
- **`read operation timed out`** (or a DNS error) — a transient network blip
  while downloading frames. Just re-run the cell: everything fetched so far is
  cached, so the retry costs seconds.

## Make it yours

1. **Your forest.** Draw a rectangle at [geojson.io](https://geojson.io) over a
   forest you know, read the corner coordinates off the map, and set
   `AOI = [west, south, east, north]`. Re-run from §4.
2. **Another index.** Swap `index="ndvi"` for `"nbr"` (burn/disturbance) or
   `"ndwi"` (water). The colorbar and labels adapt by themselves.
3. **More time.** Set `START = "2024-05-01"` and watch the frame count grow —
   one frame per month with enough cloud-free pixels.

*The seam behind all of it:* everything `animate()` accepts is one function —
[`gee_animation/api.py:163`](https://github.com/cwinkelmann/GEE_Animation/blob/main/gee_animation/api.py).
Open it: the keyword arguments are the whole public contract, and the next
notebooks do nothing you could not do from there.